# Retrieval Evaluation (Hit Rate + MRR)

This notebook evaluates vector retrieval quality with a simple and explicit protocol.

Goal:
- Create synthetic questions from each **chunk**.
- Use those questions as ground truth queries.
- Check whether the original source chunk is retrieved.


## Evaluation Design (Simple)

We keep one clear rule for relevance:
- **Relevant item = the source chunk that generated the question**.

Protocol:
1. Load chunks from Chroma collection.
2. Generate 5 synthetic questions per chunk.
3. Store them in a reusable ground-truth file.
4. Retrieve top-k chunks for each question.
5. Compute metrics:

- **Hit Rate@k** = fraction of questions where expected chunk appears in top-k.
- **MRR@k** = average reciprocal rank of the expected chunk (`1/rank` if found, else `0`).

This is intentionally chunk-level because your target is: *"are base chunks retrieved?"*


In [ ]:
from pathlib import Path
import json
import os
import random

import chromadb
from dotenv import load_dotenv
from openai import OpenAI


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
CHROMA_PATH = PROJECT_ROOT / "backend" / "data" / "chroma_db"
COLLECTION_NAME = "estate_documents"
EMBEDDING_MODEL_NAME = "text-embedding-3-small"
QUESTION_GEN_MODEL = "gpt-4o-mini"
QUESTIONS_PER_CHUNK = 5
TOP_K = 5
RANDOM_SEED = 42
MAX_CHUNKS_FOR_GT = None  # e.g. 20 for faster/cheaper iteration.
REGENERATE_GROUND_TRUTH = False
GROUND_TRUTH_PATH = PROJECT_ROOT / "backend" / "data" / "eval" / "ground_truth_chunk_questions.jsonl"

load_dotenv(PROJECT_ROOT / ".env", override=True)
if not (os.getenv("OPENAI_API_KEY") or "").strip():
    raise ValueError("OPENAI_API_KEY is missing. Add it to /workspace/.env or your environment.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Chroma path: {CHROMA_PATH}")
print(f"Collection: {COLLECTION_NAME}")
print(f"Ground truth file: {GROUND_TRUTH_PATH}")


In [ ]:
if not CHROMA_PATH.exists():
    raise FileNotFoundError(f"Chroma path does not exist: {CHROMA_PATH}")

client = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = client.get_or_create_collection(name=COLLECTION_NAME)
openai_client = OpenAI()


def fetch_all_chunks(chroma_collection, batch_size: int = 200) -> list[dict]:
    total = chroma_collection.count()
    rows: list[dict] = []

    for offset in range(0, total, batch_size):
        batch = chroma_collection.get(
            include=["documents", "metadatas"],
            limit=batch_size,
            offset=offset,
        )
        ids = batch.get("ids", [])
        docs = batch.get("documents", [])
        metas = batch.get("metadatas", [])

        for idx, chunk_id in enumerate(ids):
            rows.append(
                {
                    "chunk_id": chunk_id,
                    "text": docs[idx] or "",
                    "metadata": metas[idx] or {},
                }
            )

    return rows


all_chunks = fetch_all_chunks(collection)
print(f"Loaded {len(all_chunks)} chunk(s) from Chroma")

random.seed(RANDOM_SEED)
if MAX_CHUNKS_FOR_GT is None or MAX_CHUNKS_FOR_GT >= len(all_chunks):
    eval_chunks = list(all_chunks)
else:
    eval_chunks = random.sample(all_chunks, MAX_CHUNKS_FOR_GT)

print(f"Using {len(eval_chunks)} chunk(s) for ground-truth generation")


In [ ]:
def save_jsonl(rows: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def generate_questions_for_chunk(chunk_text: str, n_questions: int = QUESTIONS_PER_CHUNK) -> list[str]:
    snippet = chunk_text.strip()[:2500]

    system_prompt = (
        "You create short retrieval questions for a single source chunk. "
        "Questions must be answerable using only the given chunk."
    )
    user_prompt = "\n".join(
        [
            f"Generate {n_questions} diverse user-style questions based only on this chunk.",
            "Rules:",
            "- Keep questions factual and concrete.",
            "- Avoid yes/no questions.",
            "- Avoid copying full sentences from the chunk.",
            "- If names, dates, amounts, or obligations exist, include them naturally.",
            "Return strict JSON with this schema:",
            '{"questions": ["...", "...", ...]}',
            "",
            "Chunk:",
            "---",
            snippet,
            "---",
        ]
    )

    for _ in range(3):
        response = openai_client.chat.completions.create(
            model=QUESTION_GEN_MODEL,
            temperature=0.2,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        raw = response.choices[0].message.content or "{}"

        try:
            parsed = json.loads(raw)
            candidates = parsed.get("questions", [])
            cleaned = []
            seen = set()
            for q in candidates:
                if not isinstance(q, str):
                    continue
                q = q.strip()
                if not q or q in seen:
                    continue
                seen.add(q)
                cleaned.append(q)
            if len(cleaned) >= n_questions:
                return cleaned[:n_questions]
        except Exception:
            pass

    raise RuntimeError("Could not generate valid question JSON after 3 attempts.")


def build_ground_truth(chunks: list[dict], regenerate: bool = False) -> list[dict]:
    if GROUND_TRUTH_PATH.exists() and not regenerate:
        rows = load_jsonl(GROUND_TRUTH_PATH)
        print(f"Loaded existing ground truth: {len(rows)} question(s)")
        return rows

    rows: list[dict] = []
    total = len(chunks)

    for idx, chunk in enumerate(chunks, start=1):
        chunk_id = chunk["chunk_id"]
        chunk_text = chunk["text"]
        metadata = chunk.get("metadata", {})

        questions = generate_questions_for_chunk(chunk_text, n_questions=QUESTIONS_PER_CHUNK)
        for q_idx, question in enumerate(questions, start=1):
            rows.append(
                {
                    "question_id": f"{chunk_id}-q{q_idx:02d}",
                    "question": question,
                    "expected_chunk_id": chunk_id,
                    "expected_document_id": metadata.get("document_id"),
                    "document_type": metadata.get("document_type"),
                    "page_number": metadata.get("page_number"),
                }
            )

        if idx % 10 == 0 or idx == total:
            print(f"Generated questions for {idx}/{total} chunk(s)")

    save_jsonl(rows, GROUND_TRUTH_PATH)
    print(f"Saved ground truth: {len(rows)} question(s) -> {GROUND_TRUTH_PATH}")
    return rows


ground_truth_rows = build_ground_truth(
    eval_chunks,
    regenerate=REGENERATE_GROUND_TRUTH,
)
print(f"Ground-truth sample:\n{ground_truth_rows[0] if ground_truth_rows else 'No rows'}")


In [11]:
def embed_query(query: str) -> list[float]:
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL_NAME,
        input=[query],
    )
    return response.data[0].embedding


def retrieve_chunk_ids(query: str, top_k: int = TOP_K, where: dict | None = None) -> list[str]:
    query_embedding = embed_query(query)

    query_kwargs = {
        "query_embeddings": [query_embedding],
        "n_results": top_k,
        "include": ["metadatas", "distances"],
    }
    if where is not None:
        query_kwargs["where"] = where

    result = collection.query(**query_kwargs)
    return result.get("ids", [[]])[0]


def reciprocal_rank(retrieved_ids: list[str], expected_id: str) -> tuple[float, int | None]:
    for rank, chunk_id in enumerate(retrieved_ids, start=1):
        if chunk_id == expected_id:
            return 1.0 / rank, rank
    return 0.0, None


def evaluate_retrieval(ground_truth: list[dict], top_k: int = TOP_K, where: dict | None = None):
    rows: list[dict] = []

    for idx, item in enumerate(ground_truth, start=1):
        retrieved_ids = retrieve_chunk_ids(item["question"], top_k=top_k, where=where)
        rr, rank = reciprocal_rank(retrieved_ids, item["expected_chunk_id"])
        hit = int(rank is not None)

        rows.append(
            {
                **item,
                "retrieved_ids": retrieved_ids,
                "hit": hit,
                "rank": rank,
                "reciprocal_rank": rr,
            }
        )

        if idx % 50 == 0 or idx == len(ground_truth):
            print(f"Evaluated {idx}/{len(ground_truth)} question(s)")

    n = len(rows) or 1
    hit_rate = sum(row["hit"] for row in rows) / n
    mrr = sum(row["reciprocal_rank"] for row in rows) / n

    summary = {
        "num_questions": len(rows),
        "top_k": top_k,
        "hit_rate": hit_rate,
        "mrr": mrr,
    }
    return summary, rows


In [12]:
# Optional filter example:
# WHERE_FILTER = {"document_type": "mortgage_deed"}
WHERE_FILTER = None

summary, eval_rows = evaluate_retrieval(
    ground_truth_rows,
    top_k=TOP_K,
    where=WHERE_FILTER,
)

print("Evaluation summary")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"- {key}: {value:.4f}")
    else:
        print(f"- {key}: {value}")


Evaluated 50/465 question(s)
Evaluated 100/465 question(s)
Evaluated 150/465 question(s)
Evaluated 200/465 question(s)
Evaluated 250/465 question(s)
Evaluated 300/465 question(s)
Evaluated 350/465 question(s)
Evaluated 400/465 question(s)
Evaluated 450/465 question(s)
Evaluated 465/465 question(s)
Evaluation summary
- num_questions: 465
- top_k: 5
- hit_rate: 0.7140
- mrr: 0.4687


In [13]:
misses = [row for row in eval_rows if row["hit"] == 0]
print(f"Misses: {len(misses)} / {len(eval_rows)}")

for row in misses[:5]:
    print("=" * 100)
    print(f"Question: {row['question']}")
    print(f"Expected chunk: {row['expected_chunk_id']}")
    print(f"Top-{TOP_K} retrieved: {row['retrieved_ids']}")


Misses: 133 / 465
Question: Who is the notary public mentioned in the document?
Expected chunk: certificate_016-p002-c0003
Top-5 retrieved: ['donation_money_008-p003-c0004', 'donation_money_006-p003-c0005', 'donation_property_005-p003-c0004', 'purchase_003-p003-c0005', 'donation_money_007-p003-c0004']
Question: What is the purpose of the document as stated in the chunk?
Expected chunk: certificate_016-p002-c0003
Top-5 retrieved: ['purchase_017-p001-c0001', 'donation_money_006-p001-c0001', 'mortgage_014-p001-c0000', 'donation_money_007-p002-c0002', 'purchase_003-p001-c0000']
Question: Who is the notary public that executed the deed?
Expected chunk: donation_money_006-p001-c0000
Top-5 retrieved: ['donation_money_007-p003-c0004', 'donation_money_008-p003-c0004', 'purchase_003-p002-c0004', 'purchase_003-p001-c0000', 'purchase_001-p003-c0004']
Question: What is the address of the family house mentioned in the document?
Expected chunk: donation_money_006-p001-c0001
Top-5 retrieved: ['certifi